![banner](./images/banner.png)

### Suivi des feux dans les tourbières du bassin du Congo avec FIRMS

Avec ce code, vous pouvez :

0. choisir le pays (COG ou COD), voir quels jeux de données seront utilisés et afficher la carte d'occupation du sol des tourbières pour ce pays
1. télécharger les détections de feux MODIS, NOAA et SNPP de NASA FIRMS pour le pays choisi et les afficher par niveau de confiance
2. ajouter des attributs : zone administrative, aire protégée, concession forestière, classe de tourbière et distance à la tourbe (calculée à partir du raster d'occupation du sol des tourbières, sans GEE)
3. visualiser les données et une analyse préliminaire : feux à l'intérieur/à l'extérieur des concessions et des aires protégées par zone administrative, et feux par distance à la tourbe
4. exporter et télécharger sur Google Drive pour une analyse et une visualisation plus poussées
5. exporter les feux situés dans les classes de tourbière en GeoJSON pour validation dans Open Foris Ground

Toutes les analyses après l'étape 0 n'utilisent que les jeux de données du pays choisi.

In [ ]:
# en cas d'erreur avec les imports, exécuter une fois puis supprimer
#!pip install gspread --break-system-packages

In [ ]:
# en cas d'erreur avec les imports, exécuter une fois puis supprimer
#!pip install gspread google-auth google-auth-oauthlib google-auth-httplib2 --break-system-packages

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from IPython.display import display, HTML
import geopandas as gpd
from matplotlib import pyplot as plt
from matplotlib import font_manager
import geopandas as gpd
from shapely.geometry import box
import rasterio
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
from shapely.geometry import Point
import folium
from folium.plugins import HeatMap
from folium import plugins
from datetime import datetime
from datetime import datetime, date
from datetime import datetime, timedelta
import time
import gspread
import warnings
from io import StringIO
import json
from tqdm import tqdm
import shapely.geometry
from rasterio.sample import sample_gen
from pathlib import Path
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.oauth2.credentials import Credentials

# --- ajoutés pour le choix du pays, la carte des tourbières et la distance à la tourbe ---
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds
from rasterio.windows import transform as window_transform
from rasterio.windows import bounds as window_bounds
from affine import Affine
from shapely.geometry.base import BaseGeometry
from rasterio.crs import CRS
from scipy import ndimage
from scipy.spatial import cKDTree

vous devez vous inscrire et obtenir votre propre clé sur https://firms.modaps.eosdis.nasa.gov/api/map_key/ n'utilisez pas la mienne !

In [ ]:
# saisir la clé MAP_KEY reçue par e-mail - ne pas supprimer les guillemets
MAP_KEY = "1ccacc221fd0778472a2ea1e55f3e4ae"

In [ ]:
# interroger data_availability pour connaître la période disponible pour chaque jeu de données
# cette URL renvoie les informations sur tous les capteurs pris en charge et leurs jeux de données
# au lieu de 'all', vous pouvez indiquer un capteur, ex. : LANDSAT_NRT
da_url = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + MAP_KEY + '/all'
firms_data = pd.read_csv(da_url)
display(firms_data)

#### 0. Choisir le pays et les jeux de données

Indiquez `COUNTRY` = **"COG"** (République du Congo) ou **"COD"** (RD Congo), ainsi que la période et les sources FIRMS, puis exécutez le notebook de haut en bas. Toutes les étapes suivantes n'utilisent que les données de ce pays.

In [ ]:
# Pays à analyser : "COG" = République du Congo, "COD" = République démocratique du Congo
COUNTRY = "COD"

# Période des observations de feux (dates incluses) aaaa-mm-jj
start_date = "2026-01-01"
end_date   = "2026-05-01"

# Sources FIRMS : *_SP = standard/archive (périodes passées), *_NRT = quasi temps réel (~2 derniers mois)
FIRMS_SOURCES = {
    "SNPP":   "VIIRS_SNPP_SP",     # ou "VIIRS_SNPP_NRT"
    "MODIS":  "MODIS_SP",          # ou "MODIS_NRT"
    "NOAA20": "VIIRS_NOAA20_SP",   # ou "VIIRS_NOAA20_NRT"
    "NOAA21": "VIIRS_NOAA21_NRT",  # NRT uniquement (depuis 2024)
}

# ---------------------------------------------------------------------
COUNTRY_NAMES = {"COG": "République du Congo", "COD": "République démocratique du Congo"}
COUNTRY = COUNTRY.strip().upper()
if COUNTRY not in COUNTRY_NAMES:
    raise ValueError(f"COUNTRY doit être 'COG' ou 'COD', et non '{COUNTRY}'")
if datetime.strptime(start_date, "%Y-%m-%d") > datetime.strptime(end_date, "%Y-%m-%d"):
    raise ValueError("start_date doit être antérieure à end_date")
# style de tableau compact utilisé pour les résumés
def compact_table(df, title=None):
    css = ("border-collapse:collapse;font-size:12px;margin:2px 0 10px 0;")
    html = df.to_html(index=False, border=0)
    html = html.replace('<table class="dataframe">', f'<table style="{css}">')
    html = html.replace("<th>", '<th style="text-align:left;padding:3px 10px;background:#eef2f5;border-bottom:1px solid #ccc;">')
    html = html.replace("<td>", '<td style="text-align:left;padding:3px 10px;border-bottom:1px solid #eee;">')
    head = f'<div style="font-weight:bold;font-size:13px;margin-top:6px;">{title}</div>' if title else ""
    return head + html

display(HTML(compact_table(pd.DataFrame([{
    "Sélection": COUNTRY,
    "Pays": COUNTRY_NAMES[COUNTRY],
    "Du": start_date,
    "Au": end_date,
}]))))

# hauteur des cartes (pixels) : augmenter si la carte est trop petite
MAP_HEIGHT = 750

def show_map(m, height=None):
    """Affiche une carte folium dans un cadre de hauteur fixe (sans défilement)."""
    fig = folium.Figure(width="100%", height=f"{height or MAP_HEIGHT}px")
    fig.add_child(m)
    return fig

In [ ]:
# zones et périmètres à évaluer
GIS_DIR = "/home/sepal-user/Peatlands_Congo/gis_data"

adm1_path = f"{GIS_DIR}/peatlands_adm1_aoi.shp"
pas_path = f"{GIS_DIR}/peatlands_PAs.shp"
conc_cod_path = f"{GIS_DIR}/forest_concessions_COD.shp"
conc_cog_path = f"{GIS_DIR}/forest_concessions_COG.shp"

# classes d'occupation du sol des tourbières (Crezee et al. 2022)
peat_types = f"{GIS_DIR}/Crezee_2022_class_100m_4bit.tif"

# distance à la tourbe : calculée dans ce notebook à partir du raster d'occupation du sol des tourbières
PEAT_CODES = [4, 5]      # classes considérées comme tourbe : marécages dominés par les palmiers et par les feuillus
DIST_RES_M = 100         # résolution (m) de la grille utilisée pour le calcul des distances
DIST_MARGIN_KM = 50      # recherche de tourbe jusqu'à cette distance au-delà de la frontière (tourbe du pays voisin)

In [ ]:
# Classes de tourbière : communes à la carte, aux graphiques et aux tables attributaires
PEAT_CLASS_NAMES = {
    1: "eau",
    2: "savane",
    3: "forêt de terre ferme",
    4: "marécage dominé par les palmiers",
    5: "marécage dominé par les feuillus",
}
PEAT_CLASS_COLORS = {
    1: "#1f78b4",   # eau
    2: "#f5e663",   # savane
    3: "#2e8b57",   # forêt de terre ferme
    4: "#c084fc",   # marécage dominé par les palmiers
    5: "#6b21a8",   # marécage dominé par les feuillus
}

# couche des concessions et attribut conservé, par pays
CONC_DATASETS = {
    "COD": {"path": conc_cod_path, "attr": "desc_type"},
    "COG": {"path": conc_cog_path, "attr": "ste_att_en"},
}

In [ ]:
# Zone d'intérêt (AOI) des tourbières
aoi = gpd.read_file(f"{GIS_DIR}/Peatlands_AOI.shp")

In [ ]:
# ---------------------------------------------------------------------
# Charger les couches et ne garder que le pays choisi
# ---------------------------------------------------------------------
def find_col(gdf, candidates, required=True):
    """Renvoie la première colonne de gdf correspondant à l'un des candidats (sans tenir compte de la casse)."""
    lower = {c.lower(): c for c in gdf.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    if required:
        raise KeyError(f"Aucune des colonnes {candidates} trouvée. Colonnes disponibles : {list(gdf.columns)}")
    return None

def union_geom(gdf):
    try:
        return gdf.union_all()      # GeoPandas >= 0.14
    except AttributeError:
        return gdf.unary_union

# --- Zones administratives ---
ADM1_all = gpd.read_file(adm1_path).to_crs(4326)
ISO_COL = find_col(ADM1_all, ["iso3_code", "iso3", "ISO3", "adm0_iso", "GID_0"])
ADM1_NAME_COL = find_col(ADM1_all, ["gaul1_name", "adm1_name", "ADM1_NAME", "NAME_1", "name"])

ADM1 = ADM1_all[ADM1_all[ISO_COL] == COUNTRY].copy()
if ADM1.empty:
    raise ValueError(f"Aucune zone administrative avec {ISO_COL} == '{COUNTRY}' dans {adm1_path}")
COUNTRY_GEOM = union_geom(ADM1)

# --- AOI des tourbières découpée au pays (contour sur les cartes uniquement) ---
aoi = aoi.to_crs(4326)
try:
    AOI = gpd.clip(aoi, ADM1)
except Exception:
    AOI = aoi

# --- Aires protégées du pays ---
PAs_all = gpd.read_file(pas_path).to_crs(4326)
PAs = PAs_all[PAs_all.intersects(COUNTRY_GEOM)].copy()

# --- Concessions forestières du pays ---
CONC_PATH = CONC_DATASETS[COUNTRY]["path"]
CONC = gpd.read_file(CONC_PATH).to_crs(4326)
CONC = CONC[CONC.intersects(COUNTRY_GEOM)].copy()
CONC_ATTR = find_col(CONC, [CONC_DATASETS[COUNTRY]["attr"]])

# --- Emprise (bbox) des requêtes FIRMS (minx, miny, maxx, maxy) ---
minx, miny, maxx, maxy = ADM1.total_bounds
bbox = f"{minx:.4f},{miny:.4f},{maxx:.4f},{maxy:.4f}"

# ---------------------------------------------------------------------
# Résumé : fichiers SIG utilisés pour la suite de l'analyse et
# attributs qui identifient leurs entités
# ---------------------------------------------------------------------
PA_NAME_COL = find_col(PAs, ["NAME"], required=False)

def examples(gdf, col, n=6):
    if col is None or gdf.empty:
        return ""
    vals = sorted(gdf[col].dropna().astype(str).unique())
    more = f" … (+{len(vals) - n} autres)" if len(vals) > n else ""
    return ", ".join(vals[:n]) + more

def short(p):
    """gis_data/<nom du fichier> au lieu du chemin complet"""
    return f"{Path(GIS_DIR).name}/{Path(p).name}"

with rasterio.open(peat_types) as _src:
    peat_desc = f"{_src.width} x {_src.height} px, {_src.crs}"

gis_files = pd.DataFrame([
    ("Zones administratives (ADM1)", short(adm1_path),
     f"{len(ADM1)} sur {len(ADM1_all)}",
     f"{ISO_COL} = '{COUNTRY}' (filtre pays) ; {ADM1_NAME_COL} (nom de la zone administrative)",
     examples(ADM1, ADM1_NAME_COL)),
    ("Aires protégées", short(pas_path),
     f"{len(PAs)} sur {len(PAs_all)}",
     f"{PA_NAME_COL} (nom de l'aire protégée)" if PA_NAME_COL else "aucun champ NAME trouvé",
     examples(PAs, PA_NAME_COL)),
    (f"Concessions forestières ({COUNTRY})", short(CONC_PATH),
     f"{len(CONC)}",
     f"{CONC_ATTR} (attribut de la concession)",
     examples(CONC, CONC_ATTR)),
    ("AOI des tourbières (contour de carte uniquement)", short(f"{GIS_DIR}/Peatlands_AOI.shp"),
     f"{len(AOI)}", "", ""),
    ("Occupation du sol des tourbières (raster)", short(peat_types),
     peat_desc,
     "valeur du pixel = code de la classe",
     ", ".join(f"{k} {v}" for k, v in PEAT_CLASS_NAMES.items())),
    ("Distance à la tourbe (calculée)", short(peat_types),
     f"grille de {DIST_RES_M} m, pays + {DIST_MARGIN_KM} km",
     "distance au pixel le plus proche des classes de tourbe " + " / ".join(str(c) for c in PEAT_CODES),
     ", ".join(PEAT_CLASS_NAMES[c] for c in PEAT_CODES)),
], columns=["Couche", "Fichier", "Entités dans le pays", "Attribut(s) d'identification", "Valeurs (exemples)"])

firms = pd.DataFrame(
    [(k, v) for k, v in FIRMS_SOURCES.items()], columns=["Capteur", "Source FIRMS"]
)

display(HTML(
    compact_table(gis_files, f"Fichiers SIG inclus : {COUNTRY_NAMES[COUNTRY]} ({COUNTRY})")
    + compact_table(firms, "Détections de feux : capteurs disponibles")
    + f'<div style="font-size:11px;color:grey;">Zone de recherche FIRMS (bbox) : {bbox}</div>'
))

##### Occupation du sol des tourbières dans le pays choisi

In [ ]:
# (PEAT_CLASS_NAMES et PEAT_CLASS_COLORS sont définis dans la cellule des chemins ci-dessus)

MAX_PX = 2500   # taille max. de l'image affichée (le raster est sous-échantillonné pour l'affichage uniquement)

with rasterio.open(peat_types) as src:
    nodata = src.nodata
    with WarpedVRT(src, crs="EPSG:4326", resampling=Resampling.nearest) as vrt:
        # emprise du pays, limitée à l'emprise du raster
        cminx, cminy, cmaxx, cmaxy = ADM1.total_bounds
        L, B, R, T = vrt.bounds
        cminx, cminy, cmaxx, cmaxy = max(cminx, L), max(cminy, B), min(cmaxx, R), min(cmaxy, T)

        win = from_bounds(cminx, cminy, cmaxx, cmaxy, transform=vrt.transform)
        scale = max(win.width, win.height) / MAX_PX
        scale = max(scale, 1)
        out_h, out_w = int(win.height / scale), int(win.width / scale)

        peat_arr = vrt.read(1, window=win, out_shape=(out_h, out_w), resampling=Resampling.nearest)
        out_transform = window_transform(win, vrt.transform) * Affine.scale(win.width / out_w, win.height / out_h)
        left, bottom, right, top = window_bounds(win, vrt.transform)

# découpage selon la frontière du pays
inside = geometry_mask([COUNTRY_GEOM], out_shape=peat_arr.shape, transform=out_transform, invert=True)
peat_arr = np.where(inside, peat_arr, 0)
if nodata is not None:
    peat_arr[peat_arr == nodata] = 0

# coloration des classes (transparent là où il n'y a pas de classe)
rgba = np.zeros((*peat_arr.shape, 4), dtype=np.uint8)
for code_, hexcol in PEAT_CLASS_COLORS.items():
    r, g, b = [int(255 * v) for v in mcolors.to_rgb(hexcol)]
    rgba[peat_arr == code_] = [r, g, b, 220]

# --- diagramme circulaire : part de chaque classe dans le pays (à partir de l'image affichée, approximatif) ---
vals, counts = np.unique(peat_arr[peat_arr > 0], return_counts=True)
keep = [i for i, v in enumerate(vals) if int(v) in PEAT_CLASS_NAMES]
vals, counts = vals[keep], counts[keep]
pct = 100 * counts / counts.sum()
labels = [PEAT_CLASS_NAMES[int(v)] for v in vals]

fig, ax = plt.subplots(figsize=(8, 5))
wedges, _, autotexts = ax.pie(
    counts,
    colors=[PEAT_CLASS_COLORS[int(v)] for v in vals],
    autopct=lambda p: f"{p:.1f}%" if p >= 3 else "",
    pctdistance=0.75, startangle=90, counterclock=False,
    wedgeprops={"edgecolor": "white", "linewidth": 1},
)
for t, v in zip(autotexts, vals):
    t.set_color("white" if int(v) in (1, 3, 5) else "black")
    t.set_fontsize(10)
ax.legend(wedges, [f"{l} ({p:.1f}%)" for l, p in zip(labels, pct)],
          loc="center left", bbox_to_anchor=(1, 0.5), frameon=False)
ax.set_title(f"Occupation du sol des tourbières : {COUNTRY_NAMES[COUNTRY]} ({COUNTRY})")
ax.axis("equal")
plt.tight_layout()
plt.show()

# conserver l'image et son emprise pour la carte des points chauds plus loin
PEAT_RGBA = rgba
PEAT_BOUNDS = [[bottom, left], [top, right]]

# --- carte ---
m_peat = folium.Map(
    location=[(bottom + top) / 2, (left + right) / 2],
    zoom_start=6,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
)
folium.raster_layers.ImageOverlay(
    image=rgba,
    bounds=[[bottom, left], [top, right]],
    mercator_project=True,
    name=f"Classes de tourbière ({COUNTRY})",
).add_to(m_peat)

folium.GeoJson(
    ADM1[[ADM1_NAME_COL, "geometry"]], name="Zones administratives",
    style_function=lambda x: {"color": "white", "weight": 1.5, "fill": False},
    tooltip=folium.GeoJsonTooltip(fields=[ADM1_NAME_COL], aliases=["Zone administrative"]),
).add_to(m_peat)

if not PAs.empty:
    pa_name_col = find_col(PAs, ["NAME"], required=False)
    folium.GeoJson(
        PAs[[pa_name_col, "geometry"]] if pa_name_col else PAs[["geometry"]], name="Aires protégées", show=False,
        style_function=lambda x: {"color": "orange", "weight": 1.5, "fill": False},
        tooltip=folium.GeoJsonTooltip(fields=[pa_name_col]) if pa_name_col else None,
    ).add_to(m_peat)

if not CONC.empty:
    folium.GeoJson(
        CONC[[CONC_ATTR, "geometry"]], name="Concessions forestières", show=False,
        style_function=lambda x: {"color": "#7CFC00", "weight": 1, "fill": False},
        tooltip=folium.GeoJsonTooltip(fields=[CONC_ATTR]),
    ).add_to(m_peat)

legend_items = "".join(
    f'<div><span style="display:inline-block;width:12px;height:12px;background:{PEAT_CLASS_COLORS[k]};'
    f'margin-right:6px;"></span>{v}</div>'
    for k, v in PEAT_CLASS_NAMES.items()
)
m_peat.get_root().html.add_child(folium.Element(
    f'<div style="position:fixed;bottom:20px;left:10px;z-index:9999;background:white;'
    f'padding:8px 10px;border:1px solid grey;font-size:12px;">'
    f'<b>Classes de tourbière : {COUNTRY}</b>{legend_items}</div>'
))
folium.LayerControl().add_to(m_peat)
plugins.Fullscreen().add_to(m_peat)
show_map(m_peat)

#### 1. obtenir les données de feux

In [ ]:
# Colonnes FIRMS attendues (utilisées quand une source ne renvoie rien)
viirs_cols = [
    "latitude","longitude","bright_ti4","scan","track",
    "acq_date","acq_time","satellite","instrument",
    "confidence","version","bright_ti5","frp","daynight"
]
modis_cols = [
    "latitude","longitude","brightness","scan","track",
    "acq_date","acq_time","satellite","instrument",
    "confidence","version","bright_t31","frp","daynight"
]

def fetch_firms(source, label, bbox, start_date, end_date, columns, chunk=5, min_conf=30, pause=2):
    """Télécharge les détections FIRMS pour bbox de start_date à end_date (incluses).

    Les requêtes sont faites par blocs de 5 jours maximum (limite de l'API). Dans l'API FIRMS « area »,
    la date est le DÉBUT du bloc : elle renvoie [DATE .. DATE + DAY_RANGE - 1].
    Les détections de faible confiance sont supprimées (VIIRS 'l', MODIS < min_conf).
    """
    cur = datetime.strptime(start_date, "%Y-%m-%d")
    stop = datetime.strptime(end_date, "%Y-%m-%d")
    dfs = []

    while cur <= stop:
        days = min(chunk, (stop - cur).days + 1)
        date_str = cur.strftime("%Y-%m-%d")
        url = (
            f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
            f"{MAP_KEY}/{source}/{bbox}/{days}/{date_str}"
        )
        print(f"🔹 {label} : {days} jours à partir du {date_str}")

        try:
            r = requests.get(url, timeout=60)
            text = r.text.strip()

            if r.status_code != 200:
                print(f"⚠️ HTTP {r.status_code}, ignoré.")
            elif not text or "latitude" not in text.splitlines()[0]:
                print(f"⚠️ Pas de données / message de l'API : {text[:150]}")
            else:
                df = pd.read_csv(StringIO(text))
                n_all = len(df)
                if "confidence" in df.columns and not df.empty:
                    if not pd.api.types.is_numeric_dtype(df["confidence"]):   # VIIRS : l / n / h
                        df = df[~df["confidence"].astype(str).str.lower().isin(["l", "low"])]
                    else:
                        df = df[df["confidence"] >= min_conf]
                if not df.empty:
                    print(f"✅ {len(df)} enregistrements récupérés")
                    dfs.append(df)
                elif n_all:
                    print("⚠️ Uniquement des détections de faible confiance, ignorées.")
                else:
                    print("⚠️ Pas de données pour cet intervalle.")

        except Exception as e:
            print(f"❌ Échec de la requête : {e}")

        cur += timedelta(days=days)
        time.sleep(pause)

    if dfs:
        out = pd.concat(dfs, ignore_index=True).drop_duplicates()
    else:
        out = pd.DataFrame(columns=columns)
    out["satellite"] = label
    print(f"\n🔥 {label} : {len(out)} enregistrements au total")
    return out

In [ ]:
all_SNPP = fetch_firms(FIRMS_SOURCES["SNPP"], "SNPP", bbox, start_date, end_date, viirs_cols)

In [ ]:
all_SNPP.head()

In [ ]:
all_MODIS = fetch_firms(FIRMS_SOURCES["MODIS"], "MODIS", bbox, start_date, end_date, modis_cols)

In [ ]:
all_MODIS.head()

In [ ]:
all_NOAA20 = fetch_firms(FIRMS_SOURCES["NOAA20"], "NOAA20", bbox, start_date, end_date, viirs_cols)

In [ ]:
all_NOAA21 = fetch_firms(FIRMS_SOURCES["NOAA21"], "NOAA21", bbox, start_date, end_date, viirs_cols)

##### combiner toutes les détections de feux

In [ ]:
# Combiner
merged_fires = pd.concat([all_MODIS, all_NOAA21, all_NOAA20, all_SNPP], ignore_index=True)

merged_fires_gdf = gpd.GeoDataFrame(
    merged_fires,
    geometry=gpd.points_from_xy(merged_fires.longitude, merged_fires.latitude),
    crs="EPSG:4326"
)
len(merged_fires_gdf)

In [ ]:
# supprimer les doublons
# S'assurer que acq_date est bien au format date
merged_fires_gdf["acq_date"] = pd.to_datetime(merged_fires_gdf["acq_date"])

# Trier par date décroissante (la plus récente en premier)
merged_fires_gdf = merged_fires_gdf.sort_values("acq_date", ascending=False)

# Supprimer les doublons par coordonnées en gardant la plus récente
fires_unique = merged_fires_gdf.drop_duplicates(subset=["latitude", "longitude"], keep="first")

print(f"🔥 Détections de feux uniques : {len(fires_unique)}")

#### 2. extraire et ajouter les attributs

In [ ]:
# Couches déjà chargées et filtrées sur le pays choisi à l'étape 0
layers = {
    "ADM1": ADM1,
    "PAs": PAs,
    f"CONC_{COUNTRY}": CONC,
}

for name, gdf in layers.items():
    print(f"\nChamps attributaires de {name} ({len(gdf)} entités)")
    print("-" * (len(name) + 36))
    for col in [c for c in gdf.columns if c != "geometry"]:
        print(f"• {col}")

In [ ]:
# Ne garder que les feux situés dans les zones administratives du pays choisi
merged_fires_gdf = merged_fires_gdf.to_crs(ADM1.crs)

inside_idx = gpd.sjoin(
    merged_fires_gdf[["geometry"]], ADM1[["geometry"]], how="inner", predicate="within"
).index.unique()
fires_within_adm1 = merged_fires_gdf.loc[inside_idx].copy()

print(f"🔥 Feux dans les zones administratives de {COUNTRY} : {len(fires_within_adm1):,}")

In [ ]:
# --- Passer en WGS84 pour Folium ---
fires_within_adm1 = fires_within_adm1.to_crs(epsg=4326)

# S'assurer que la colonne existe
if "confidence_num" not in fires_within_adm1.columns:
    fires_within_adm1["confidence_num"] = 1.0
else:
    fires_within_adm1["confidence_num"] = fires_within_adm1["confidence_num"].fillna(1.0)

# --- Gérer un jeu de données vide ---
if fires_within_adm1.empty:
    center_lat, center_lon = 0, 0
else:
    center_lat = fires_within_adm1.geometry.y.mean()
    center_lon = fires_within_adm1.geometry.x.mean()

# --- Fond de carte sombre ---
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=6,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, HERE, Garmin, © OpenStreetMap contributors",
    max_zoom=16,
)

# --- 🔥 Préparer les poids des points chauds de feux ---
if "confidence" in fires_within_adm1.columns:

    fires_within_adm1["confidence_num"] = pd.to_numeric(
        fires_within_adm1["confidence"], errors="coerce"
    )

    # Si les valeurs de confiance sont catégorielles (type VIIRS)
    if fires_within_adm1["confidence_num"].isna().all():

        conf_map = {"low": 0.5, "nominal": 1.0, "medium": 1.0, "high": 2.0}

        fires_within_adm1["confidence_num"] = (
            fires_within_adm1["confidence"]
            .astype(str)
            .str.lower()
            .map(conf_map)
            .fillna(1.0)
        )

    else:
        fires_within_adm1["confidence_num"] = fires_within_adm1["confidence_num"].fillna(1.0)

else:
    fires_within_adm1["confidence_num"] = 1.0

# --- Construire la liste de données des points chauds de feux ---
heat_data = [
    [geom.y, geom.x, float(conf)]
    for geom, conf in zip(
        fires_within_adm1.geometry,
        fires_within_adm1["confidence_num"]
    )
    if geom is not None
]

# --- Ajouter les points chauds de feux seulement s'il y a des données ---
if heat_data:
    plugins.HeatMap(
        heat_data,
        radius=8,
        blur=15,
        max_zoom=9,
        min_opacity=0.3,
    ).add_to(m)

# --- Outils de carte optionnels ---
plugins.Fullscreen().add_to(m)
plugins.MiniMap(toggle_display=True).add_to(m)
plugins.LocateControl(auto_start=False).add_to(m)

show_map(m)

##### ajouter les classes de tourbière à partir du raster dans SEPAL

In [ ]:
with rasterio.open(peat_types) as src:
    # Aligner le SCR sur celui du raster
    fires_within_adm1 = fires_within_adm1.to_crs(src.crs)

    coords = [(pt.x, pt.y) for pt in fires_within_adm1.geometry]

    batch_size = 50000
    peat_vals = []

    nodata = src.nodata  # peut être None

    for i in tqdm(range(0, len(coords), batch_size), desc="Échantillonnage"):
        batch_coords = coords[i:i + batch_size]

        for v in src.sample(batch_coords):
            val = v[0]  # première bande

            # Convertir nodata en NaN
            if nodata is not None and val == nodata:
                peat_vals.append(np.nan)
            else:
                peat_vals.append(val)

assert len(peat_vals) == len(fires_within_adm1), "Nombre de valeurs échantillonnées incorrect."

# Ajouter les résultats (tous les feux sont conservés ; peat_class_code peut être NaN)
fires_within_adm1["peat_class_code"] = peat_vals
fires_within_adm1["has_peat_class"] = fires_within_adm1["peat_class_code"].notna().astype(int)

In [ ]:
# Convertir les codes numériques en libellés (PEAT_CLASS_NAMES est défini dans la cellule des chemins)
fires_within_adm1["peat_class_name"] = fires_within_adm1["peat_class_code"].map(PEAT_CLASS_NAMES)

# Remplacer les valeurs manquantes ou nulles par "pas de données"
fires_within_adm1["peat_class_name"] = fires_within_adm1["peat_class_name"].fillna("pas de données")
fires_within_adm1.loc[
    fires_within_adm1["peat_class_code"] == 0,
    "peat_class_name"
] = "pas de données"

# --- Supprimer les feux sans classe de tourbière (hors du raster ou nodata) ---
n_before = len(fires_within_adm1)
fires_within_adm1 = fires_within_adm1[fires_within_adm1["peat_class_name"] != "pas de données"].copy()
print(f"{n_before - len(fires_within_adm1):,} feux sans classe de tourbière supprimés ; {len(fires_within_adm1):,} restants")
fires_within_adm1["peat_class_name"].value_counts()

In [ ]:
warnings.filterwarnings("ignore", message="Glyph 128293")
# --- Nombre de feux par classe de tourbière (NaN inclus) ---
peat_counts = (
    fires_within_adm1["peat_class_name"]
    .value_counts(dropna=False)
    .rename_axis("Classe")
    .reset_index(name="Nombre de feux")
)
peat_counts["Classe"] = peat_counts["Classe"].fillna("pas de données")

# --- Mêmes couleurs que la carte des tourbières ---
color_map = {name: PEAT_CLASS_COLORS[code_] for code_, name in PEAT_CLASS_NAMES.items()}
color_map["pas de données"] = "grey"
peat_counts["Couleur"] = peat_counts["Classe"].str.lower().map(color_map).fillna("grey")

# --- Graphique ---
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(
    peat_counts["Classe"],
    peat_counts["Nombre de feux"],
    color=peat_counts["Couleur"]
)
ax.bar_label(bars, labels=[f"{v:,}" for v in peat_counts["Nombre de feux"]],
             label_type="edge", padding=3, fontsize=10, color="black")

ax.set_title(f"Nombre de feux par classe de tourbière : {COUNTRY}", fontsize=14)
ax.set_xlabel("Classe de tourbière")
ax.set_ylabel("Nombre de feux")
plt.xticks(rotation=30, ha="right")
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

##### calculer la distance à la tourbe

Pour chaque feu, distance (km) au pixel de tourbe le plus proche (classes de `PEAT_CODES`, par défaut 4 marécage dominé par les palmiers et 5 marécage dominé par les feuillus) dans le raster d'occupation du sol des tourbières. Les feux situés sur la tourbe reçoivent 0.

Fonctionnement : le raster est rééchantillonné sur une grille de `DIST_RES_M` dans une projection conservant les distances, centrée sur le pays et couvrant le pays plus `DIST_MARGIN_KM`, afin de tenir compte de la tourbe située de l'autre côté de la frontière. Les pixels de bordure des taches de tourbe sont extraits et chaque feu est associé au pixel de bordure le plus proche. Le raster des tourbières étant à 100 m, `DIST_RES_M = 100` utilise sa pleine résolution ; la précision est d'environ ±50 m.

In [ ]:
# Classes de distance (km). La première classe (0) = feu sur la tourbe
DIST_BINS = [-0.001, 0, 1, 2, 5, 10, 25, np.inf]
DIST_LABELS = ["0 (sur tourbe)", "0-1 km", "1-2 km", "2-5 km", "5-10 km", "10-25 km", "> 25 km"]

# --- projection conservant les distances, centrée sur le pays (azimutale équidistante, mètres) ---
c = COUNTRY_GEOM.centroid
DIST_CRS = CRS.from_proj4(f"+proj=aeqd +lat_0={c.y:.4f} +lon_0={c.x:.4f} +datum=WGS84 +units=m +no_defs")

# --- grille couvrant le pays + la marge ---
gminx, gminy, gmaxx, gmaxy = ADM1.to_crs(DIST_CRS).total_bounds
m_ = DIST_MARGIN_KM * 1000
gminx, gminy, gmaxx, gmaxy = gminx - m_, gminy - m_, gmaxx + m_, gmaxy + m_
gw = int(np.ceil((gmaxx - gminx) / DIST_RES_M))
gh = int(np.ceil((gmaxy - gminy) / DIST_RES_M))
grid_transform = Affine(DIST_RES_M, 0, gminx, 0, -DIST_RES_M, gmaxy)
print(f"Grille de distance : {gw:,} x {gh:,} px à {DIST_RES_M} m")

with rasterio.open(peat_types) as src:
    with WarpedVRT(src, crs=DIST_CRS, transform=grid_transform, width=gw, height=gh,
                   resampling=Resampling.nearest) as vrt:
        grid = vrt.read(1)

# --- masque de tourbe et pixels de bordure des taches de tourbe ---
peat_mask = np.isin(grid, PEAT_CODES)
del grid
edge = peat_mask & ~ndimage.binary_erosion(peat_mask)
rows, cols = np.nonzero(edge)
del edge
print(f"Pixels de tourbe : {peat_mask.sum():,}  |  pixels de bordure : {len(rows):,}")

fires_within_adm1 = fires_within_adm1.to_crs(4326)

if len(rows) == 0:
    print("⚠️ Pas de tourbe dans la grille : distance mise à NaN")
    dist_m = np.full(len(fires_within_adm1), np.nan)
else:
    # centres des pixels de bordure (m) -> arbre KD -> pixel de bordure le plus proche de chaque feu
    ex = grid_transform.c + (cols + 0.5) * grid_transform.a
    ey = grid_transform.f + (rows + 0.5) * grid_transform.e
    tree = cKDTree(np.column_stack([ex, ey]))

    fp = fires_within_adm1.to_crs(DIST_CRS)
    fxy = np.column_stack([fp.geometry.x, fp.geometry.y])
    dist_m, _ = tree.query(fxy, workers=-1)

    # feux sur la tourbe = 0 : d'après la classe échantillonnée à pleine résolution ci-dessus,
    # ou d'après la grille (feu dans un pixel de tourbe de la grille)
    r = np.floor((fxy[:, 1] - grid_transform.f) / grid_transform.e).astype(int)
    k = np.floor((fxy[:, 0] - grid_transform.c) / grid_transform.a).astype(int)
    in_grid = (r >= 0) & (r < gh) & (k >= 0) & (k < gw)
    on_peat_grid = np.zeros(len(fxy), dtype=bool)
    on_peat_grid[in_grid] = peat_mask[r[in_grid], k[in_grid]]
    on_peat = fires_within_adm1["peat_class_code"].isin(PEAT_CODES).values | on_peat_grid
    dist_m = np.where(on_peat, 0.0, dist_m)

del peat_mask

fires_within_adm1["dist_peat_km"] = np.round(dist_m / 1000, 3)
fires_within_adm1["dist_class"] = (
    pd.cut(fires_within_adm1["dist_peat_km"], bins=DIST_BINS, labels=DIST_LABELS)
    .astype(str).replace("nan", "pas de données")
)

print(fires_within_adm1["dist_peat_km"].describe().round(2))

# Contrôle : classe de distance vs classe de tourbière
display(HTML("<div style='font-weight:bold;font-size:13px;margin:6px 0 2px 0;'>Nombre de feux par distance à la tourbe</div>"))
display(pd.crosstab(fires_within_adm1["peat_class_name"], fires_within_adm1["dist_class"])
        .reindex(columns=[l for l in DIST_LABELS + ["pas de données"] if l in set(fires_within_adm1["dist_class"])]))

In [ ]:
# ------------------------------------------------------------------
# Joindre les attributs avec un préfixe.
# Un feu situé sur une limite commune ou dans des polygones superposés serait
# dupliqué par la jointure ; seule la première correspondance est gardée
# pour que chaque feu ne soit compté qu'une fois.
# ------------------------------------------------------------------
def spatial_join_all_prefix(base_gdf, join_gdf, prefix, predicate="intersects"):
    join_gdf = join_gdf.copy()
    join_gdf.drop(columns=["Shape_Area", "Shape_Leng"], errors="ignore", inplace=True)

    joined = gpd.sjoin(base_gdf, join_gdf, how="left", predicate=predicate)
    joined = joined[~joined.index.duplicated(keep="first")]

    join_cols = [c for c in join_gdf.columns if c != "geometry"]
    rename_dict = {c: f"{prefix}_{c}" for c in join_cols if c in joined.columns}
    joined.rename(columns=rename_dict, inplace=True)

    joined.drop(columns=["index_right"], errors="ignore", inplace=True)
    return joined

def spatial_join_some_prefix(base_gdf, join_gdf, columns, prefix, predicate="intersects"):
    join_gdf = join_gdf.copy()

    # Correspondance des colonnes demandées sans tenir compte de la casse
    col_map = {c.lower(): c for c in join_gdf.columns}
    matched = [col_map[c.lower()] for c in columns if c.lower() in col_map]

    join_gdf = join_gdf[["geometry"] + matched].copy()

    joined = gpd.sjoin(base_gdf, join_gdf, how="left", predicate=predicate)
    joined = joined[~joined.index.duplicated(keep="first")]

    rename_dict = {c: f"{prefix}_{c}" for c in matched if c in joined.columns}
    joined.rename(columns=rename_dict, inplace=True)

    joined.drop(columns=["index_right"], errors="ignore", inplace=True)
    return joined

fires_within_adm1 = fires_within_adm1.to_crs(ADM1.crs)

# ------------------------------------------------------------------
# Appliquer les jointures (les couches sont déjà limitées au pays choisi)
# ------------------------------------------------------------------
fires_all_atts = spatial_join_all_prefix(fires_within_adm1, ADM1, "ADM")
fires_all_atts = spatial_join_some_prefix(fires_all_atts, PAs, ["NAME"], "PA")
fires_all_atts = spatial_join_some_prefix(fires_all_atts, CONC, [CONC_ATTR], "for_conc")

ADM_COL = f"ADM_{ADM1_NAME_COL}"          # nom de la zone administrative de chaque feu
CONC_COL = f"for_conc_{CONC_ATTR}"        # attribut de concession de chaque feu

# ------------------------------------------------------------------
# Champs indicateurs
# ------------------------------------------------------------------
fires_all_atts["PA"] = np.where(fires_all_atts.get("PA_NAME").notna(), 1, 0)
fires_all_atts["for_conc"] = np.where(fires_all_atts[CONC_COL].notna(), 1, 0)

fires_all_atts.head(5)

In [ ]:
print("\n".join(fires_all_atts.columns))

#### 3. Évaluer et visualiser

In [ ]:
fire_counts = fires_all_atts["satellite"].value_counts().reset_index()
fire_counts.columns = ["satellite", "Nombre de feux"]

print("🔥 Détections de feux par source :")
print(fire_counts)

fires_all_atts["acq_date"] = pd.to_datetime(fires_all_atts["acq_date"], errors="coerce")

daily_counts = fires_all_atts.groupby(["acq_date", "satellite"]).size().reset_index(name="Nombre de feux")

warnings.filterwarnings("ignore", message="Glyph .* missing from font")

plt.figure(figsize=(14, 6))
for src in daily_counts["satellite"].unique():
    data = daily_counts[daily_counts["satellite"] == src]
    plt.plot(data["acq_date"], data["Nombre de feux"], marker="o", markersize=3, label=src)

plt.title("Détections de feux quotidiennes par satellite")
plt.xlabel("Date")
plt.ylabel("Nombre de feux")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Type de détection")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# -------------------------------------------------------
# Convertir la date d'acquisition
# -------------------------------------------------------
fires_all_atts["acq_date"] = pd.to_datetime(
    fires_all_atts["acq_date"], errors="coerce"
)

# -------------------------------------------------------
# Harmoniser les valeurs de confiance
# -------------------------------------------------------
fires_all_atts["confidence_num"] = pd.to_numeric(
    fires_all_atts["confidence"], errors="coerce"
)

# Confiance numérique (MODIS)
fires_all_atts["confidence_class"] = pd.cut(
    fires_all_atts["confidence_num"],
    bins=[-np.inf, 30, 80, np.inf],
    labels=["Faible", "Nominale", "Élevée"],
    right=False
)

# Remplacer la confiance catégorielle VIIRS
fires_all_atts.loc[
    fires_all_atts["confidence"].astype(str).str.lower().isin(["l", "low"]),
    "confidence_class"
] = "Faible"

fires_all_atts.loc[
    fires_all_atts["confidence"].astype(str).str.lower().isin(["n", "nominal"]),
    "confidence_class"
] = "Nominale"

fires_all_atts.loc[
    fires_all_atts["confidence"].astype(str).str.lower().isin(["h", "high"]),
    "confidence_class"
] = "Élevée"

# -------------------------------------------------------
# Paramètres des graphiques
# -------------------------------------------------------
confidence_order = ["Faible", "Nominale", "Élevée"]

colors = {
    "Faible": "#fdbb84",
    "Nominale": "orange",
    "Élevée": "red"
}

satellites = fires_all_atts["satellite"].dropna().unique()

warnings.filterwarnings("ignore", message="Glyph .* missing from font")

# -------------------------------------------------------
# Graphique par satellite
# -------------------------------------------------------
for sat in satellites:

    sat_df = fires_all_atts[fires_all_atts["satellite"] == sat]

    # Nombre de feux par date + confiance
    daily_conf_counts = (
        sat_df
        .groupby(["acq_date", "confidence_class"], observed=True)
        .size()
        .reset_index(name="Nombre de feux")
    )

    pivot_df = (
        daily_conf_counts
        .pivot(index="acq_date", columns="confidence_class", values="Nombre de feux")
        .fillna(0)
    )

    pivot_df = pivot_df.reindex(
        columns=[c for c in confidence_order if c in pivot_df.columns]
    )

    # ---------------------------------------------------
    # Barres empilées
    # ---------------------------------------------------
    fig, ax = plt.subplots(figsize=(14, 6))

    bottom = pd.Series(0, index=pivot_df.index)

    for conf in pivot_df.columns:
        ax.bar(
            pivot_df.index,
            pivot_df[conf],
            bottom=bottom,
            color=colors.get(conf, "gray"),
            label=conf
        )
        bottom += pivot_df[conf]

    # ---------------------------------------------------
    # Mise en forme des axes
    # ---------------------------------------------------
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))

    plt.xticks(rotation=45, ha="right")

    plt.title(f"Détections de feux quotidiennes par niveau de confiance — {sat}")
    plt.xlabel("Date")
    plt.ylabel("Nombre de feux")

    plt.legend(
        title="Confiance",
        bbox_to_anchor=(1.05, 1),
        loc="upper left"
    )

    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Feux sur la tourbe (PEAT_CODES : 4 marécage à palmiers, 5 marécage à feuillus) ---
fires_peat = fires_all_atts[fires_all_atts["peat_class_code"].isin(PEAT_CODES)].copy()
print(f"Feux sur la tourbe : {len(fires_peat):,} sur {len(fires_all_atts):,} en {COUNTRY}")

def inside_outside_by_adm(df, flag_col):
    t = (
        pd.crosstab(df[ADM_COL].fillna("non attribué"), df[flag_col].map({0: "Extérieur", 1: "Intérieur"}))
        .reindex(columns=["Intérieur", "Extérieur"], fill_value=0)
    )
    t["Total"] = t["Intérieur"] + t["Extérieur"]
    t["% intérieur"] = (100 * t["Intérieur"] / t["Total"]).round(1)
    return t.sort_values("Total")

def with_total(t):
    t = t.sort_values("Total", ascending=False)
    tot = t[["Intérieur", "Extérieur", "Total"]].sum()
    tot["% intérieur"] = round(100 * tot["Intérieur"] / tot["Total"], 1) if tot["Total"] else 0.0
    out = pd.concat([t, tot.to_frame(f"TOTAL {COUNTRY}").T])
    return out.astype({"Intérieur": int, "Extérieur": int, "Total": int})

if fires_peat.empty:
    print("⚠️ Aucun feu dans les classes de tourbe pour ce pays et cette période.")
else:
    conc_by_adm = inside_outside_by_adm(fires_peat, "for_conc")
    pa_by_adm = inside_outside_by_adm(fires_peat, "PA")

    n_adm = len(conc_by_adm)
    fig, axes = plt.subplots(1, 2, figsize=(16, max(4, 0.5 * n_adm + 2)))

    for ax, tab, title, colour in [
        (axes[0], conc_by_adm, "des concessions forestières", "darkgreen"),
        (axes[1], pa_by_adm, "des aires protégées", "orange"),
    ]:
        tab[["Intérieur", "Extérieur"]].plot(
            kind="barh", stacked=True, ax=ax, color=[colour, "lightgrey"], width=0.75
        )
        ax.bar_label(
            ax.containers[-1],
            labels=[f"{t:,} ({p:.0f} % à l'intérieur)" for t, p in zip(tab["Total"], tab["% intérieur"])],
            padding=3, fontsize=9,
        )
        ax.set_title(f"Feux sur la tourbe à l'intérieur / à l'extérieur {title}\npar zone administrative : {COUNTRY}")
        ax.set_xlabel("Nombre de feux")
        ax.set_ylabel("")
        ax.margins(x=0.3)
        ax.grid(axis="x", linestyle="--", alpha=0.4)
        ax.legend(loc="lower right")

    plt.tight_layout()
    plt.show()

    print("Concessions forestières, feux sur la tourbe par zone administrative :")
    display(with_total(conc_by_adm))
    print("Aires protégées, feux sur la tourbe par zone administrative :")
    display(with_total(pa_by_adm))

##### Feux par distance à la tourbe

In [ ]:
dist_order = DIST_LABELS + ["pas de données"]
dfd = fires_all_atts.copy()
dfd["dist_class"] = pd.Categorical(dfd["dist_class"], categories=dist_order, ordered=True)

# couleurs : rouge foncé = sur ou près de la tourbe, clair = loin
cmap = plt.get_cmap("YlOrRd_r")
dist_colors = [cmap(i / (len(DIST_LABELS))) for i in range(len(DIST_LABELS))] + ["lightgrey"]

# --- 1. Tous les feux du pays par classe de distance + courbe cumulée ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

counts = dfd["dist_class"].value_counts().reindex(dist_order, fill_value=0)
bars = axes[0].bar(counts.index.astype(str), counts.values, color=dist_colors)
axes[0].bar_label(bars, labels=[f"{v:,}" for v in counts.values], padding=3, fontsize=9)
axes[0].set_title(f"Feux par distance à la tourbe : {COUNTRY}")
axes[0].set_ylabel("Nombre de feux")
axes[0].tick_params(axis="x", rotation=30)
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

d = np.sort(dfd["dist_peat_km"].dropna().values)
if len(d):
    axes[1].plot(d, 100 * np.arange(1, len(d) + 1) / len(d), color="firebrick")
    axes[1].set_xlim(0, min(50, d.max()) if d.max() > 0 else 1)
    summary_lines = []
    for km in [1, 5, 10]:
        pct = 100 * (d <= km).mean()
        axes[1].axvline(km, color="grey", linestyle=":", lw=1)
        axes[1].plot(km, pct, "o", color="firebrick", ms=4)
        summary_lines.append(f"≤ {km:>2} km : {pct:4.0f} % des feux")
    axes[1].text(0.97, 0.05, "\n".join(summary_lines), transform=axes[1].transAxes,
                 ha="right", va="bottom", fontsize=9, family="monospace",
                 bbox=dict(boxstyle="round", facecolor="white", edgecolor="lightgrey"))
axes[1].set_title("Part cumulée des feux selon la distance à la tourbe")
axes[1].set_xlabel("Distance à la tourbe (km)")
axes[1].set_ylabel("% des feux")
axes[1].grid(linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

# --- 2. Par zone administrative ---
dist_by_adm = pd.crosstab(dfd[ADM_COL].fillna("non attribué"), dfd["dist_class"], dropna=False)
dist_by_adm = dist_by_adm.reindex(columns=dist_order, fill_value=0)
dist_by_adm = dist_by_adm.loc[dist_by_adm.sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(13, max(4, 0.5 * len(dist_by_adm) + 2)))
dist_by_adm.plot(kind="barh", stacked=True, ax=ax, color=dist_colors, width=0.75)
ax.set_title(f"Feux par distance à la tourbe et par zone administrative : {COUNTRY}")
ax.set_xlabel("Nombre de feux")
ax.set_ylabel("")
ax.legend(title="Distance à la tourbe", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

# --- 3. Tableau : effectifs par classe + distance médiane et part à moins de 1 / 5 km par zone administrative ---
stats = dfd.groupby(dfd[ADM_COL].fillna("non attribué"))["dist_peat_km"].agg(
    mediane_km="median",
    pct_moins_1km=lambda s: 100 * (s <= 1).mean(),
    pct_moins_5km=lambda s: 100 * (s <= 5).mean(),
).round(1)
dist_table = dist_by_adm.join(stats)
dist_table.insert(len(dist_order), "Total", dist_by_adm.sum(axis=1))
display(HTML("<div style='font-weight:bold;font-size:13px;margin:6px 0 2px 0;'>Nombre de feux par distance à la tourbe et par zone administrative</div>"))
display(dist_table.sort_values("Total", ascending=False))

In [ ]:
# visualiser sur une carte
# --- S'assurer que les colonnes latitude/longitude existent ---
if not {"latitude", "longitude"}.issubset(fires_all_atts.columns):
    fires_all_atts["latitude"] = fires_all_atts.geometry.y
    fires_all_atts["longitude"] = fires_all_atts.geometry.x

# --- Préparer les données des points chauds de feux ---
heat_data = fires_all_atts[["latitude", "longitude"]].dropna().values.tolist()

# --- Créer la carte ---
m = folium.Map(
    location=[fires_all_atts.latitude.mean(), fires_all_atts.longitude.mean()],
    zoom_start=6,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery"
)

# --- Classes de tourbière (étape 0), ajoutées sous les points chauds de feux ---
folium.raster_layers.ImageOverlay(
    image=PEAT_RGBA,
    bounds=PEAT_BOUNDS,
    mercator_project=True,
    opacity=0.6,
    name=f"Classes de tourbière ({COUNTRY})",
).add_to(m)

# --- Dégradé personnalisé bleu → rouge ---
custom_gradient = {
    0.0: "blue",
    0.25: "cyan",
    0.5: "lime",
    0.75: "yellow",
    1.0: "red"
}

# --- Ajouter les points chauds de feux ---
HeatMap(
    heat_data,
    name="Points chauds de feux",
    radius=6,
    blur=10,
    min_opacity=0.4,
    gradient=custom_gradient
).add_to(m)

# --- Ajouter les limites ADM1 (sans remplissage, contours blancs) ---
if "ADM1" in locals() and ADM1 is not None:
    folium.GeoJson(
        ADM1,
        name="Limites ADM1",
        style_function=lambda x: {
            "color": "white",
            "weight": 1,
            "fill": False
        }
    ).add_to(m)

# --- Ajouter l'AOI (optionnel) ---
if "AOI" in locals() and AOI is not None:
    folium.GeoJson(
        AOI,
        name="AOI",
        style_function=lambda x: {"color": "blue", "weight": 2, "fill": False}
    ).add_to(m)

# --- Ajouter la légende ---
legend_html = """
<div style="
position: fixed;
bottom: 10px; left: 10px; width: 200px; height: 90px;
background-color: white; border:2px solid grey; z-index:9999; font-size:13px;
padding: 10px;">
<b>🔥 Densité de feux</b><br>
<svg width="180" height="15">
  <defs>
    <linearGradient id="grad" x1="0%" y1="0%" x2="100%" y2="0%">
      <stop offset="0%" style="stop-color:blue;stop-opacity:1" />
      <stop offset="25%" style="stop-color:cyan;stop-opacity:1" />
      <stop offset="50%" style="stop-color:lime;stop-opacity:1" />
      <stop offset="75%" style="stop-color:yellow;stop-opacity:1" />
      <stop offset="100%" style="stop-color:red;stop-opacity:1" />
    </linearGradient>
  </defs>
  <rect width="180" height="15" fill="url(#grad)" />
</svg><br>
Faible&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Élevée
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# --- Légende des classes de tourbière (en bas à droite) ---
peat_legend_items = "".join(
    f'<div><span style="display:inline-block;width:12px;height:12px;background:{PEAT_CLASS_COLORS[k]};'
    f'opacity:0.8;margin-right:6px;"></span>{v}</div>'
    for k, v in PEAT_CLASS_NAMES.items()
)
m.get_root().html.add_child(folium.Element(
    f'<div style="position:fixed;bottom:10px;right:10px;z-index:9999;background:white;'
    f'padding:8px 10px;border:2px solid grey;font-size:12px;">'
    f'<b>Classes de tourbière : {COUNTRY}</b>{peat_legend_items}</div>'
))

# --- Contrôle des couches ---
folium.LayerControl().add_to(m)

show_map(m)

In [ ]:
# les feux sont déjà limités au pays choisi (étape 0)
fires_all_atts_filt = fires_all_atts.copy()
print(f"Nombre de feux en {COUNTRY} :", len(fires_all_atts_filt))

#### 4. télécharger les résultats sur Google Drive

In [ ]:
# autoriser le téléchargement sur Google Drive
cred_path = Path.home() / ".config/earthengine/credentials"
with open(cred_path, "r") as f:
    access_token = json.load(f).get("access_token")
drive = build('drive', 'v3', credentials=Credentials(token=access_token))

In [ ]:
# exporter en CSV
output_path = f"/home/sepal-user/Peatlands_Congo/fires_{COUNTRY}_{start_date}_{end_date}.csv"
# Exporter d'abord en CSV dans SEPAL
fires_all_atts_filt.to_csv(output_path, index_label='ID')

In [ ]:
# S'assurer que le dossier Peatlands existe
folder_name = "Congo_Peatlands"
folder_results = drive.files().list(
    q=f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false",
    fields="files(id, name)"
).execute()
folders = folder_results.get("files", [])

if not folders:
    print(f"⚠️ Dossier '{folder_name}' introuvable. Création en cours...")
    folder_metadata = {"name": folder_name, "mimeType": "application/vnd.google-apps.folder"}
    folder = drive.files().create(body=folder_metadata, fields="id").execute()
    folder_id = folder["id"]
    print(f"🆕 Dossier '{folder_name}' créé (ID : {folder_id})")
else:
    folder_id = folders[0]["id"]
    print(f"📁 Dossier '{folder_name}' trouvé (ID : {folder_id})")

In [ ]:
# nommer le fichier de sortie
SHEET_NAME = f"{COUNTRY}_Fire_Monitoring_{start_date}_{end_date}"

In [ ]:
fires_all_atts_filt.to_csv(output_path, index_label="ID")

print(f"📁 CSV exporté localement : {output_path}")

# -------------------------------------------------------
# Téléchargement sur Google Drive
# -------------------------------------------------------
VIIRS_file_metadata = {
    "name": SHEET_NAME.replace(".csv", ""),
    "mimeType": "application/vnd.google-apps.spreadsheet",
    "parents": [folder_id]
}

media = MediaFileUpload(output_path, mimetype="text/csv", resumable=True)

# -------------------------------------------------------
# Vérifier si le fichier existe déjà
# -------------------------------------------------------
existing = drive.files().list(
    q=f"name='{SHEET_NAME.replace('.csv','')}' and '{folder_id}' in parents and trashed=false",
    fields="files(id,name)"
).execute()

files = existing.get("files", [])

# -------------------------------------------------------
# Remplacer ou télécharger
# -------------------------------------------------------
if files:
    file_id = files[0]["id"]
    print(f"♻️ Remplacement de la feuille Google existante '{SHEET_NAME}'...")

    updated = drive.files().update(
        fileId=file_id,
        media_body=media
    ).execute()

    file_id = updated["id"]

else:
    uploaded = drive.files().create(
        body=VIIRS_file_metadata,
        media_body=media,
        fields="id"
    ).execute()

    file_id = uploaded["id"]

print(f"✅ '{SHEET_NAME}' téléchargé dans le dossier Google Drive '{folder_name}'")

# -------------------------------------------------------
# Afficher un grand lien cliquable
# -------------------------------------------------------
sheet_url = f"https://docs.google.com/spreadsheets/d/{file_id}/edit"

display(HTML(f"""
<div style="margin-top:20px;">
<b>Ouvrir les données de feux dans Google</b><br>
<a href="{sheet_url}" target="_blank"
style="font-size:22px;font-weight:bold;color:#1a73e8;">
{sheet_url}
</a>
</div>
"""))

#### 5. exporter les feux dans les tourbières en GeoJSON pour Ground

In [ ]:
# --- Filtrer les feux sur la tourbe (PEAT_CODES) ---
fires_peat = fires_all_atts[
    fires_all_atts["peat_class_code"].isin(PEAT_CODES)
].copy()

print(f"✅ {len(fires_peat):,} feux sélectionnés sur la tourbe, classes {PEAT_CODES} ({COUNTRY})")

# --- Garder peat_class_name, supprimer le code numérique ---
fires_peat.drop(columns=["peat_class_code"], inplace=True, errors="ignore")

# --- Ajouter la propriété 'name' à partir de la date d'acquisition ---
fires_peat["name"] = fires_peat["acq_date"].astype(str).apply(
    lambda d: f"Feu_{d[:10]}" if pd.notna(d) and len(d) >= 10 else "Feu_inconnu"
)

# colonnes catégorielles -> texte pour le pilote GeoJSON
for c in fires_peat.columns:
    if isinstance(fires_peat[c].dtype, pd.CategoricalDtype):
        fires_peat[c] = fires_peat[c].astype(str)

today_str = datetime.today().strftime("%Y%m%d")

if fires_peat.empty:
    print(f"⚠️ Aucun feu sur la tourbe pour {COUNTRY} entre {start_date} et {end_date} : aucun fichier GeoJSON exporté.")
else:
    output_path = f"fires_peatland_{COUNTRY}_{today_str}.geojson"
    fires_peat.to_crs(4326).to_file(output_path, driver="GeoJSON")
    print(f"💾 {len(fires_peat):,} feux exportés pour {COUNTRY} → {output_path}")